In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [2]:
df = pd.read_excel('cetes.xlsx')
df

,Fecha,SF43936
0,2020-01-02,7.25
1,2020-01-09,7.26
2,2020-01-16,7.00
3,2020-01-23,7.05
4,2020-01-30,7.04
...,...,...
328,2026-04-16,6.60
329,2026-04-23,6.55
330,2026-04-30,6.50
331,2026-05-07,6.49


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 333 entries, 0 to 332
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Fecha    333 non-null    datetime64[ns]
 1   SF43936  333 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 5.3 KB


# Estimación Modelo de Black
## $dr_t = \alpha dt + \sigma dW_t$ 
### Parámetros a estimar: $\alpha$ y $\sigma$

In [4]:
df['tasa en %'] = df['SF43936'] / 100

In [5]:
df['Cambios semanales'] = df['tasa en %'].diff()

In [6]:
df.head()

,Fecha,SF43936,tasa en %,Cambios semanales
0,2020-01-02,7.25,0.0725,NaN
1,2020-01-09,7.26,0.0726,0.0001
2,2020-01-16,7.00,0.0700,-0.0026
3,2020-01-23,7.05,0.0705,0.0005
4,2020-01-30,7.04,0.0704,-0.0001


In [7]:
promedio_cambios = df['Cambios semanales'][:len(df)-1].mean()

In [8]:
# Estimar alpha
delta_t = 1/52
alpha = promedio_cambios / delta_t
print(f'alpha={alpha}')

alpha=-0.0011939577039274916


In [9]:
desv_cambios = df['Cambios semanales'][:len(df)-1].std(ddof=1)

In [10]:
sigma = desv_cambios / np.sqrt(delta_t)
print(f'sigma={sigma}')

sigma=0.010097927468342677


# Estimación modelo de Rendleman-Batter
## $dr_t = \mu r_t dt + \sigma r_t dW_t$ 
### Parámetros a estimar: $\mu$ y $\sigma$

In [11]:
df['log_return'] = np.log(df['tasa en %'] / df['tasa en %'].shift(1))
df.head()

,Fecha,SF43936,tasa en %,Cambios semanales,log_return
0,2020-01-02,7.25,0.0725,NaN,NaN
1,2020-01-09,7.26,0.0726,0.0001,0.001378
2,2020-01-16,7.00,0.0700,-0.0026,-0.036470
3,2020-01-23,7.05,0.0705,0.0005,0.007117
4,2020-01-30,7.04,0.0704,-0.0001,-0.001419


In [12]:
#sigma
sigma = df['log_return'][:len(df)-1].std(ddof=1) * np.sqrt(delta_t)
print(f'sigma={sigma}')

sigma=0.002594712378987028


In [13]:
# mu
mu = (df['log_return'][:len(df)-1].mean() / delta_t) + ((1/2) * sigma**2)
print(f'mu={mu}')

mu=-0.01739368746141955


# Estimación Modelo de Vasicek
## $dr_t = a(b - r_t)dt + \sigma dW_t$ 
### Parámetros a estimar: a, b y $\sigma$

In [14]:
df['delta_r'] = df['tasa en %'].shift(-1) - df['tasa en %']

In [15]:
X = df['tasa en %'][:-1]
y = df['delta_r'][:-1]

In [16]:
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                delta_r   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.003
Method:                 Least Squares   F-statistic:                    0.1551
Date:                Thu, 14 May 2026   Prob (F-statistic):              0.694
Time:                        14:01:09   Log-Likelihood:                 1711.5
No. Observations:                 332   AIC:                            -3419.
Df Residuals:                     330   BIC:                            -3411.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       7.098e-05      0.000      0.288      0.7

In [17]:
alpha_hat = model.params['const']
beta_hat = model.params['tasa en %']

In [18]:
a_hat = -beta_hat / delta_t
b_hat = alpha_hat / (a_hat * delta_t)

In [19]:
residuos = model.resid
sigma_hat = residuos.std(ddof=1) / np.sqrt(delta_t)

In [20]:
print(f'a = {a_hat}, b = {b_hat}, sigma = {sigma_hat}')

a = 0.06137406553078901, b = 0.06013623432065643, sigma = 0.010082416954693387
